# Auto Evidence 360: Real-Source Quality Review

## tl;dr

- The current downloaded snapshot contains **1,411,783 real public-data rows** across seven analytical sources.
- All seven files matched their documented field counts; the current profile found no malformed rows.
- Exact normalized make/model/year matching ranges from **15.47% to 64.86%**, proving that controlled entity resolution is a central project requirement.
- Public complaint PII-like fields and narratives are excluded from Fabric upload extracts.
- Complaint and bulletin volume are evidence signals, not make/model reliability rates.

## Context & Methods

The business question is simple: which make/model/year combinations deserve deeper review before a used vehicle is bought or listed?

### Key Assumptions

- A public record is evidence that an event or filing exists, not proof that every covered vehicle is defective.
- Cross-source comparisons require a transparent vehicle identity bridge.
- Counts without a make/model/year exposure denominator must not be labeled failure or reliability rates.
- The notebook emits aggregate checks only. It never displays complaint narratives, VIN fragments, contact fields, cities, or vehicle-operator fields.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config" / "sources.json").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "config" / "sources.json").exists(), "Project root could not be resolved"
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "analysis" / "profile_real_sources.py")],
    cwd=PROJECT_ROOT,
    check=True,
)
profile_path = PROJECT_ROOT / "analysis" / "output" / "source_profile.json"
profile = json.loads(profile_path.read_text(encoding="utf-8"))
profile["generated_at_utc"]

Profiling nhtsa_complaints...


Profiling nhtsa_recalls...


Profiling nhtsa_investigations...


Profiling nhtsa_manufacturer_communications...


Profiling nhtsa_ncap...
Profiling epa_fuel_economy...


Profiling fhwa_state_registrations...
{
  "json": "/Users/harivinayak/Codex/Car IQ/fabric-auto-intelligence/analysis/output/source_profile.json",
  "markdown": "/Users/harivinayak/Codex/Car IQ/fabric-auto-intelligence/analysis/output/source_profile.md"
}


'2026-08-17T19:51:01.874852+00:00'

## Data

Each source is downloaded from the publisher URL in `config/sources.json`. The downloader writes retrieval metadata and a SHA-256 checksum beside every file. Official NHTSA dictionaries define the tab-delimited schemas.

In [2]:
dataset_profile = pd.DataFrame(profile["datasets"])
display_columns = [
    "source_id", "rows", "columns", "malformed_rows", "exact_duplicate_rows",
    "distinct_normalized_vehicle_keys", "min_year", "max_year", "min_source_date", "max_source_date"
]
dataset_profile[display_columns].fillna("n/a")

,source_id,rows,columns,malformed_rows,exact_duplicate_rows,distinct_normalized_vehicle_keys,min_year,max_year,min_source_date,max_source_date
0,nhtsa_complaints,182995,51,0,0,6923,1986,2027,20250101,20260813
1,nhtsa_recalls,244398,29,0,0,39956,1965,2027,20100101,20260810
2,nhtsa_investigations,154249,11,0,0,14844,1965,2026,19720310,20260731
3,nhtsa_manufacturer_communications,731898,14,0,1527,19472,2000,2027,20241204,20260815
4,nhtsa_ncap,17313,128,0,1,12282,1990,2026,n/a,n/a
5,epa_fuel_economy,50242,84,0,0,26359,1984,2027,n/a,n/a
6,fhwa_state_registrations,30688,5,0,0,0,1900,2024,n/a,n/a


## Results

The first result checks whether the source files can be parsed at their documented shape. The second quantifies the entity-resolution gap before any alias or token matching is allowed.

In [3]:
total_rows = int(dataset_profile["rows"].fillna(0).sum())
total_malformed = int(dataset_profile["malformed_rows"].fillna(0).sum())
pd.DataFrame({
    "metric": ["Downloaded analytical rows", "Malformed rows", "Sources profiled"],
    "value": [total_rows, total_malformed, int((dataset_profile["status"] == "profiled").sum())],
})

,metric,value
0,Downloaded analytical rows,1411783
1,Malformed rows,0
2,Sources profiled,7


In [4]:
match_coverage = pd.DataFrame(profile["cross_source_exact_match_coverage"])
match_coverage.assign(exact_match_rate=lambda frame: frame["exact_match_rate"].map(lambda value: f"{value:.2%}"))

,source_id,distinct_vehicle_keys,exact_reference_matches,exact_match_rate
0,nhtsa_complaints,6923,4490,64.86%
1,nhtsa_recalls,39956,6182,15.47%
2,nhtsa_investigations,14844,4980,33.55%
3,nhtsa_manufacturer_communications,19472,6473,33.24%


## Takeaways

1. **The data are large enough for a credible Fabric project.** Complexity comes from multiple grains and schemas, not fabricated volume.
2. **Entity resolution is measurable work.** Exact matching is the high-confidence baseline; aliases and token rules need reviewed mappings and coverage reporting.
3. **Repeated business IDs can be legitimate.** A campaign or bulletin can repeat for multiple vehicles or components, so semantic measures must count the correct business entity.
4. **Privacy minimization happens before cloud upload.** Only approved fields in `data/fabric_upload/` enter Fabric.
5. **The dashboard must use precise language.** It reports public-record signals and review priority, not a universal safety or reliability ranking.